# Domain 1: Agentic Architecture & Orchestration
## Task Statement 1.6: Design Task Decomposition Strategies for Complex Workflows
**Certification Weight**: 27% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand when to use **Fixed Prompt Chaining** (Multi-Pass Reviews) versus **Dynamic Adaptive Decomposition**, and how to eliminate attention dilution across complex codebases and documents.

---

## 🏗️ Architecture: Two-Pass Review vs Single-Pass Dilution

```mermaid
flowchart TD
    subgraph AntiPattern["Anti-Pattern: Single-Pass Monolith"]
        RawFiles["5 Target Source Files"] --> SinglePrompt["Single Claude Turn<br>(Prompt: Review all 5 files)"]
        SinglePrompt --> DilutionResult["Diluted Focus:<br>Misses cross-file contract breaks,<br>shallow file review"]
    end

    subgraph TwoPassWorkflow["Recommended: Two-Pass Chaining"]
        Files["Target Source Files"] --> Pass1["Pass 1: Isolated File Review<br>(Evaluate syntax, local logic, null safety)"]
        Pass1 --> Findings["Structured File Finding Cards"]
        Findings --> Pass2["Pass 2: Integration & Contract Review<br>(Evaluate API signatures, data flow, shared types)"]
        Pass2 --> FinalReport["Comprehensive High-Precision Report"]
    end
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Strategy | When to Select on the Exam | Exam Trap to Reject |
|---|---|---|
| **Two-Pass Sequential Chaining** | Fixed multi-file reviews or security audits where local syntax and cross-file contracts both matter. | **TRAP**: Combining file inspection and integration analysis into a single giant prompt ("Review this entire repo in one prompt"). |
| **Dynamic Adaptive Decomposition** | Open-ended workflows where downstream steps depend on exploratory findings (e.g. repo migration, dependency analysis). | **TRAP**: Hardcoding rigid sequential subtask lists upfront for exploratory, unpredictable workflows. |
| **Attention Dilution Avoidance** | Bounding the scope of each pass to either single-file syntax OR cross-module contract validation. | **TRAP**: Stuffing multiple disparate objectives (formatting, security, performance, architecture) into a single agent turn. |

---

## 💡 Architectural Decision Matrix

| Task Nature | Decomposition Strategy | Structure | SLA / Latency Profile |
|---|---|---|---|
| **Deterministic Quality Gates** | Fixed Multi-Pass Chaining | Pass 1 (Local) $\rightarrow$ Pass 2 (Contract) | Predictable step count, higher token spend, near-zero missed bugs. |
| **Open-Ended Architecture Tasks** | Adaptive Tree-of-Thought / Subtasking | Coordinator dynamically generates subtasks | Variable turn count, adaptive to repo complexity. |
| **Simple Narrow Inquiries** | Single Turn with Tools | 1 Loop Iteration | Lowest latency and token cost. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.6 Task Decomposition.")

### 📂 Step 2: Code Representation & Two-Pass Pipeline
We define code files and implement the two-pass prompt chaining workflow.

In [ ]:
# Step 2: Two-Pass Decomposition Logic
@dataclass
class CodeFile:
    filename: str
    content: str

def run_prompt_chaining_review(files: List[CodeFile], client: Any) -> Dict[str, Any]:
    """
    Two-Pass Review Pipeline:
    Pass 1: Isolated per-file review
    Pass 2: Cross-file integration synthesis
    """
    local_findings = {}

    # --- PASS 1: Isolated Per-File Analysis ---
    print("=== Starting Pass 1: Local File Analysis ===")
    for f in files:
        prompt = f"Analyze file {f.filename} in isolation for syntax, bugs, and security risks:\n{f.content}"
        resp = client.messages.create(
            model="claude-haiku-4-5",
            messages=[{"role": "user", "content": prompt}],
        )
        finding_text = "".join(b.text for b in resp.content if b.type == "text")
        local_findings[f.filename] = finding_text
        print(f"  [Pass 1] {f.filename}: {finding_text}")

    # --- PASS 2: Cross-File Integration Analysis ---
    print("\n=== Starting Pass 2: Cross-File Integration Analysis ===")
    pass2_prompt = (
        "Synthesize the following file-level findings and evaluate cross-file contracts:\n"
        + json.dumps(local_findings, indent=2)
    )
    resp2 = client.messages.create(
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": pass2_prompt}],
    )
    integration_findings = "".join(b.text for b in resp2.content if b.type == "text")
    print(f"  [Pass 2] Integration Summary: {integration_findings}")

    return {
        "local_findings": local_findings,
        "integration_findings": integration_findings,
    }

print(" Two-pass decomposition pipeline ready.")

### 🧪 Step 3: Offline Simulation Walkthrough
We review two interacting files:
- `auth_service.py`: Generates JWT tokens.
- `api_router.py`: Verifies tokens and queries database.
Pass 1 checks each file individually. Pass 2 identifies that the router expects token expiration but the service does not encode it.

In [ ]:
# Step 3: Simulation with Mock Responses
files_to_review = [
    CodeFile("auth_service.py", "def generate_token(user_id): return jwt.encode({'user': user_id})"),
    CodeFile("api_router.py", "def route(request): token = request.headers['auth']; verify(token, check_exp=True)"),
]

# Mock Pass 1 (file 1, file 2) and Pass 2 (integration)
mock_resp1 = MockMessageResponse(content=[MockContentBlock(block_type="text", text="auth_service lacks expiry timestamp in JWT.")])
mock_resp2 = MockMessageResponse(content=[MockContentBlock(block_type="text", text="api_router strictly requires check_exp=True.")])
mock_resp_cross = MockMessageResponse(content=[MockContentBlock(block_type="text", text="CRITICAL CONTRACT MISMATCH: api_router fails token verification because auth_service omits exp field.")])

client = MockClaudeClient([mock_resp1, mock_resp2, mock_resp_cross])
results = run_prompt_chaining_review(files_to_review, client)

### 🔬 Step 4: Verification Assertions
We verify that:
1. Pass 1 generated isolated per-file findings.
2. Pass 2 caught the cross-file contract mismatch that neither isolated pass could catch alone.

In [ ]:
# Step 4: Verification Checks
assert "lacks expiry" in results["local_findings"]["auth_service.py"]
assert "CONTRACT MISMATCH" in results["integration_findings"]

print(" All verification assertions passed! Decomposition strategy proven.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Avoid feeding large multi-file diffs into a single monolithic prompt (attention dilution).
- **Rule 2**: Structure complex reviews as a **Two-Pass Prompt Chain**: local per-file checks followed by a cross-file contract synthesis pass.
- **Rule 3**: Contract mismatches (parameter names, missing fields) are discovered in Pass 2.